# Bronze Layer Ingestion

## Import Libraries and Start Session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName("olist-bronze-setup")
        .getOrCreate()
)

# Load the Spark SQL magic python file created earlier
from spark_sql_magic import sql  # registers the %%sql magic automatically

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


# Test CDC Ingestion Once

## Read from Kafka

In [2]:
# BATCH LOAD CDC table ONCE to check the code.

df_kafka = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "olist.oltp.customers")
    .option("startingOffsets", "earliest")
    .load()
)

In [3]:
df_kafka.printSchema()
# we need value, partition, offset and timestamp

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## Ingestion Schema: Fetch via REST API

In [6]:
import requests

response = requests.get(
    "http://apicurio:8081/apis/registry/v2/groups/default/artifacts/olist.oltp.customers-value/versions/latest",
    timeout = 10
)
response.raise_for_status()

schema_dict = response.json()
schema_dict

{'type': 'record',
 'name': 'Value',
 'namespace': 'olist.oltp.customers',
 'fields': [{'name': 'customer_id', 'type': 'string'},
  {'name': 'customer_unique_id', 'type': ['null', 'string'], 'default': None},
  {'name': 'customer_zip_code_prefix',
   'type': ['null', 'string'],
   'default': None},
  {'name': 'customer_city', 'type': ['null', 'string'], 'default': None},
  {'name': 'customer_state', 'type': ['null', 'string'], 'default': None},
  {'name': 'created_at',
   'type': [{'type': 'long',
     'connect.version': 1,
     'connect.default': 0,
     'connect.name': 'io.debezium.time.MicroTimestamp'},
    'null'],
   'default': 0},
  {'name': 'updated_at',
   'type': [{'type': 'long',
     'connect.version': 1,
     'connect.default': 0,
     'connect.name': 'io.debezium.time.MicroTimestamp'},
    'null'],
   'default': 0},
  {'name': '__op', 'type': ['null', 'string'], 'default': None},
  {'name': '__ts_ms', 'type': ['null', 'long'], 'default': None}],
 'connect.name': 'olist.olt

## Dynamic Table Creation w/ Avro Schema

### Create a mapping dictionanry

In [ ]:
from pyspark.sql.types import *

AVRO_TO_SPARK = {
    "string": StringType(),
    "long": LongType(),
    "int": IntegerType(),
    "boolean": BooleanType(),
    "double": DoubleType(),
    "float": FloatType()
}

### Function: Extract Spark Type for each row

In [ ]:
def resolve_avro_type(avro_type):
    """
    Returns (spark_type, nullable)
    """

    nullable = False

    # Case 1: List ('type': ['null', 'string'])
    if isinstance(avro_type, list):
        nullable = "null" in avro_type
        avro_type = [t for t in avro_type if t != "null"][0]

    # Case 2: Dict 
    """
    'type': [{'type': 'long',
     'connect.version': 1,
     'connect.default': 0,
     'connect.name': 'io.debezium.time.MicroTimestamp'},
    'null']
    """
    if isinstance(avro_type, dict):
        base_type = avro_type["type"]
        if avro_type.get("connect.name") == "io.debezium.time.MicroTimestamp":
            return TimestampType(), True
        return AVRO_TO_SPARK.get(base_type, StringType()), True

    # Case 3: Simple type
    return AVRO_TO_SPARK.get(avro_type, StringType()), nullable

### Dynamically Build Schema (StructType) for each table

In [ ]:
def build_spark_schema(schema_dict):
    fields = []

    for field in schema_dict["fields"]:
        name = field["name"]
        avro_type = field["type"]

        spark_type, nullable = resolve_avro_type(avro_type)

        fields.append(StructField(name, spark_type, nullable))

    return StructType(fields)

In [7]:
# convert to json string
schema_json = json.dumps(schema_dict)
schema_json

'{"type": "record", "name": "Value", "namespace": "olist.oltp.customers", "fields": [{"name": "customer_id", "type": "string"}, {"name": "customer_unique_id", "type": ["null", "string"], "default": null}, {"name": "customer_zip_code_prefix", "type": ["null", "string"], "default": null}, {"name": "customer_city", "type": ["null", "string"], "default": null}, {"name": "customer_state", "type": ["null", "string"], "default": null}, {"name": "created_at", "type": [{"type": "long", "connect.version": 1, "connect.default": 0, "connect.name": "io.debezium.time.MicroTimestamp"}, "null"], "default": 0}, {"name": "updated_at", "type": [{"type": "long", "connect.version": 1, "connect.default": 0, "connect.name": "io.debezium.time.MicroTimestamp"}, "null"], "default": 0}, {"name": "__op", "type": ["null", "string"], "default": null}, {"name": "__ts_ms", "type": ["null", "long"], "default": null}], "connect.name": "olist.oltp.customers.Value"}'

## Generate a unique batch_id with python

In [8]:
# this is needed for Spark streaming foreachBatch
import datetime, uuid

batch_id = f"{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}-{uuid.uuid4().hex[:8]}"

## Deserialize/Parse Avro

In [9]:
from pyspark.sql.avro.functions import from_avro

df_parsed = (
    df_kafka
        .select(
            from_avro("value", schema_json).alias("data"),
            F.col("offset"),
            F.col("partition"),
            F.col("timestamp").alias("kafka_ingest_ts")
        )
        .select("data.*", "offset", "partition", "kafka_ingest_ts")
        .withColumnRenamed("created_at", "pg_created_at")
        .withColumnRenamed("updated_at", "pg_updated_at")
        .withColumnRenamed("__ts_ms", "pg_commit_ts")
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("spark_ingest_ts", F.current_timestamp())
)

In [10]:
df_parsed.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- pg_created_at: long (nullable = true)
 |-- pg_updated_at: long (nullable = true)
 |-- __op: string (nullable = true)
 |-- pg_commit_ts: long (nullable = true)
 |-- offset: long (nullable = true)
 |-- partition: integer (nullable = true)
 |-- kafka_ingest_ts: timestamp (nullable = true)
 |-- batch_id: string (nullable = false)
 |-- spark_ingest_ts: timestamp (nullable = false)



## Write into Bronze Test Table

In [11]:
# https://iceberg.apache.org/docs/latest/spark-structured-streaming/
# write > writeStream
# add checkpoint s3a://olist-ecommerce/checkpoints/bronze/customers/
# mode > outputMode
# saveAsTable > toTable

(
    df_parsed.write
    .format("iceberg")
    .mode("append")
    .saveAsTable("polaris.bronze.test_customers")
)

[13:00:41] [Thread-3] INFO  org.apache.iceberg.rest.auth.AuthManagers - Loading AuthManager implementation: org.apache.iceberg.rest.auth.OAuth2Manager
[13:00:42] [Thread-3] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO
[13:00:43] [Thread-3] INFO  org.apache.iceberg.rest.RESTSessionCatalog - Table properties set at catalog level through catalog properties: {}
[13:00:43] [Thread-3] INFO  org.apache.iceberg.rest.RESTSessionCatalog - Table properties enforced at catalog level through catalog properties: {}
[13:00:44] [Thread-3] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO


[13:00:52] [Thread-3] INFO  org.apache.iceberg.spark.source.SparkWrite - Committing append with 3 new data files to table polaris.bronze.test_customers
[13:00:52] [Thread-3] INFO  org.apache.iceberg.SnapshotProducer - Committed snapshot 5586251883378256945 (MergeAppend)
[13:00:52] [Thread-3] INFO  org.apache.iceberg.metrics.LoggingMetricsReporter - Received metrics report: CommitReport{tableName=polaris.bronze.test_customers, snapshotId=5586251883378256945, sequenceNumber=1, operation=append, commitMetrics=CommitMetricsResult{totalDuration=TimerResult{timeUnit=NANOSECONDS, totalDuration=PT0.53590595S, count=1}, attempts=CounterResult{unit=COUNT, value=1}, addedDataFiles=CounterResult{unit=COUNT, value=3}, removedDataFiles=null, totalDataFiles=CounterResult{unit=COUNT, value=3}, addedDeleteFiles=null, addedEqualityDeleteFiles=null, addedPositionalDeleteFiles=null, addedDVs=null, removedDeleteFiles=null, removedEqualityDeleteFiles=null, removedPositionalDeleteFiles=null, removedDVs=null,

In [12]:
%%sql
DROP TABLE IF EXISTS polaris.bronze.test_customers;

[13:01:17] [Thread-3] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO


## Define Ingestion Configuration

In [13]:
INGESTION_CONFIG = [
    "customers",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers"
]

In [14]:
queries = []

for item in INGESTION_CONFIG:
    print(f"Ingesting topic olist.oltp.{item} → polaris.bronze.{item}")

    df_kafka = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9092")
        .option("subscribe", f"olist.oltp.{item}")
        .option("startingOffsets", "earliest")
        .load()
    )

    response = requests.get(
        f"http://apicurio:8081/apis/registry/v2/groups/default/artifacts/olist.oltp.{item}-value/versions/latest",
        timeout = 10
    )
    response.raise_for_status()

    schema_dict = response.json()
    schema_json = json.dumps(schema_dict)

    df_parsed = (
        df_kafka
            .select(
                from_avro("value", schema_json).alias("data"),
                F.col("offset"),
                F.col("partition"),
                F.col("timestamp").alias("kafka_ingest_ts")
            )
            .select("data.*", "offset", "partition", "kafka_ingest_ts")
            .withColumnRenamed("created_at", "pg_created_at")
            .withColumnRenamed("updated_at", "pg_updated_at")
            .withColumnRenamed("__ts_ms", "pg_commit_ts")
    )

    def build_write_batch(table_name):
        def write_batch(batch_df, epoch_id):
    
            import uuid, datetime
        
            batch_id = f"{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}-{uuid.uuid4().hex[:8]}"

            writer = (
                batch_df
                .withColumn("batch_id", F.lit(batch_id))
                .withColumn("spark_ingest_ts", F.current_timestamp())
                .writeTo(table_name)
                .option("merge-schema", "true")
            )
            
            if not spark.catalog.tableExists(table_name):
                writer.create()
            else:
                writer.append()
        return write_batch

    
    query = (
        df_parsed.writeStream
        .foreachBatch(build_write_batch(f"polaris.bronze.{item}"))
        .option("checkpointLocation", f"s3a://olist-ecommerce/checkpoints/bronze/{item}/")
        .start()
    )

    queries.append(query)

# try:
#     spark.streams.awaitAnyTermination()
# except Exception:
#     print("Stream failure detected. Shutting down...")
    
#     for q in spark.streams.active:
#         q.stop()
    
#     raise

print(f"Started {len(queries)} streams. Use queries[i].stop() to stop a stream manually.")

Ingesting topic olist.oltp.customers → polaris.bronze.customers
[13:01:29] [Thread-3] WARN  org.apache.hadoop.metrics2.impl.MetricsConfig - Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
[13:01:29] [Thread-3] WARN  org.apache.spark.sql.execution.streaming.ResolveWriteToStream - spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
Ingesting topic olist.oltp.order_items → polaris.bronze.order_items
[13:01:30] [Thread-3] WARN  org.apache.spark.sql.execution.streaming.ResolveWriteToStream - spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
Ingesting topic olist.oltp.order_payments → polaris.bronze.order_payments
[13:01:31] [Thread-3] WARN  org.apache.spark.sql.execution.streaming.ResolveWriteToStream - spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
Ingesting topic olist.oltp.order_reviews

[Stage 1:>                  (0 + 3) / 3][Stage 2:>                  (0 + 3) / 3]

[13:01:32] [stream execution thread for [id = 2412805e-7024-4fdb-a5bd-9c4c43ed4abb, runId = f6ae111c-a360-4471-bd36-dc1e016df5cb]] INFO  org.apache.iceberg.rest.auth.AuthManagers - Loading AuthManager implementation: org.apache.iceberg.rest.auth.OAuth2Manager
[13:01:32] [stream execution thread for [id = 2412805e-7024-4fdb-a5bd-9c4c43ed4abb, runId = f6ae111c-a360-4471-bd36-dc1e016df5cb]] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO
[13:01:32] [stream execution thread for [id = 2412805e-7024-4fdb-a5bd-9c4c43ed4abb, runId = f6ae111c-a360-4471-bd36-dc1e016df5cb]] INFO  org.apache.iceberg.rest.RESTSessionCatalog - Table properties set at catalog level through catalog properties: {}
[13:01:32] [stream execution thread for [id = 2412805e-7024-4fdb-a5bd-9c4c43ed4abb, runId = f6ae111c-a360-4471-bd36-dc1e016df5cb]] INFO  org.apache.iceberg.rest.RESTSessionCatalog - Table properties enforced at catalog level through catalog prope

[Stage 1:>    (0 + 3) / 3][Stage 2:>    (0 + 3) / 3][Stage 3:>    (0 + 3) / 3]

[13:01:32] [Thread-3] WARN  org.apache.spark.sql.execution.streaming.ResolveWriteToStream - spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
Ingesting topic olist.oltp.sellers → polaris.bronze.sellers
[13:01:33] [stream execution thread for [id = c774d40d-a38e-4ccf-bd47-0a3b2f20e40e, runId = 5825660a-035a-4661-9cf2-ffd62b08cef9]] INFO  org.apache.iceberg.rest.auth.AuthManagers - Loading AuthManager implementation: org.apache.iceberg.rest.auth.OAuth2Manager
[13:01:33] [stream execution thread for [id = c774d40d-a38e-4ccf-bd47-0a3b2f20e40e, runId = 5825660a-035a-4661-9cf2-ffd62b08cef9]] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO
[13:01:33] [stream execution thread for [id = c774d40d-a38e-4ccf-bd47-0a3b2f20e40e, runId = 5825660a-035a-4661-9cf2-ffd62b08cef9]] INFO  org.apache.iceberg.rest.RESTSessionCatalog - Table properties set at catalog level through catalog properties

[13:01:35] [stream execution thread for [id = f99821be-863e-4069-9a48-02d2b6f58c7e, runId = 7b5c4f85-9129-4187-bf08-535c110a5ca3]] INFO  org.apache.iceberg.spark.source.SparkWrite - Committing append with 3 new data files to table polaris.bronze.order_payments
[13:01:35] [stream execution thread for [id = 180e2654-a3c6-40a9-ab0e-fa00680f0023, runId = 6b595bca-98ff-4167-844a-7e2592abead0]] INFO  org.apache.iceberg.rest.auth.AuthManagers - Loading AuthManager implementation: org.apache.iceberg.rest.auth.OAuth2Manager
[13:01:35] [stream execution thread for [id = 180e2654-a3c6-40a9-ab0e-fa00680f0023, runId = 6b595bca-98ff-4167-844a-7e2592abead0]] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO
[13:01:35] [stream execution thread for [id = f99821be-863e-4069-9a48-02d2b6f58c7e, runId = 7b5c4f85-9129-4187-bf08-535c110a5ca3]] INFO  org.apache.iceberg.SnapshotProducer - Committed snapshot 824191628233547812 (MergeAppend)
[13:01:35

[13:01:36] [stream execution thread for [id = 180e2654-a3c6-40a9-ab0e-fa00680f0023, runId = 6b595bca-98ff-4167-844a-7e2592abead0]] INFO  org.apache.iceberg.spark.source.SparkWrite - Committing append with 3 new data files to table polaris.bronze.sellers
[13:01:37] [stream execution thread for [id = 3ab51c03-570e-4fb1-ba92-90d3063cfb7a, runId = 47f8be8a-a34f-4537-b51a-cb6853230e5b]] INFO  org.apache.iceberg.spark.source.SparkWrite - Committing append with 3 new data files to table polaris.bronze.products
[13:01:37] [stream execution thread for [id = 180e2654-a3c6-40a9-ab0e-fa00680f0023, runId = 6b595bca-98ff-4167-844a-7e2592abead0]] INFO  org.apache.iceberg.SnapshotProducer - Committed snapshot 1770351276230449726 (MergeAppend)
[13:01:37] [stream execution thread for [id = 180e2654-a3c6-40a9-ab0e-fa00680f0023, runId = 6b595bca-98ff-4167-844a-7e2592abead0]] INFO  org.apache.iceberg.metrics.LoggingMetricsReporter - Received metrics report: CommitReport{tableName=polaris.bronze.sellers, sn

## Sanity Check Show All Tables

In [15]:
%%sql
SHOW TABLES IN bronze;

+---------+--------------+-----------+
|namespace|tableName     |isTemporary|
+---------+--------------+-----------+
|bronze   |customers     |false      |
|bronze   |order_payments|false      |
|bronze   |orders        |false      |
|bronze   |sellers       |false      |
|bronze   |order_reviews |false      |
|bronze   |order_items   |false      |
|bronze   |products      |false      |
+---------+--------------+-----------+



In [16]:
for item in INGESTION_CONFIG:
    df = spark.table(f"polaris.bronze.{item}").limit(5)
    df.show()

[13:01:54] [Thread-3] INFO  org.apache.iceberg.CatalogUtil - Loading custom FileIO implementation: org.apache.iceberg.aws.s3.S3FileIO
[13:01:54] [Thread-3] INFO  org.apache.iceberg.SnapshotScan - Scanning table polaris.bronze.customers snapshot 2011499220051369983 created at 2026-02-15T13:01:34.819+00:00 with filter true
[13:01:54] [Thread-3] INFO  org.apache.iceberg.BaseDistributedDataScan - Planning file tasks locally for table polaris.bronze.customers
[13:01:54] [Thread-3] INFO  org.apache.iceberg.metrics.LoggingMetricsReporter - Received metrics report: ScanReport{tableName=polaris.bronze.customers, snapshotId=2011499220051369983, filter=true, schemaId=0, projectedFieldIds=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], projectedFieldNames=[customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state, pg_created_at, pg_updated_at, __op, pg_commit_ts, offset, partition, kafka_ingest_ts, batch_id, spark_ingest_ts], scanMetrics=ScanMetricsResult{totalPlann

+--------------------+--------------------+------------------------+-------------+--------------+----------------+----------------+----+-------------+------+---------+--------------------+--------------------+--------------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|   pg_created_at|   pg_updated_at|__op| pg_commit_ts|offset|partition|     kafka_ingest_ts|            batch_id|     spark_ingest_ts|
+--------------------+--------------------+------------------------+-------------+--------------+----------------+----------------+----+-------------+------+---------+--------------------+--------------------+--------------------+
|0002414f953443074...|4893ad4ea28b2c5b3...|                   39664|     mendonca|            MG|1766340931955560|1766340931955560|   r|1771147104441|     0|        0|2026-02-15 09:18:...|20260215130131-00...|2026-02-15 13:01:...|
|0004164d20a9e969a...|104bdb7e6a6cdceaa...|                   13272|     val

In [17]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  